[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# Criterios y *trade-offs* de endurecimiento.

El endurecimiento no es gratis. Cada control añadido reduce una superficie de ataque pero aumenta la complejidad operativa, el tiempo de ciclo o la fragilidad del sistema. Este cuaderno propone un marco para decidir *cuánto* endurecer y *dónde* — una decisión que en la práctica es tan técnica como estratégica.

## Objetivos.

- Nombrar el costo real de cada control de seguridad implementado en el curso.
- Identificar los puntos de fragilidad que el endurecimiento introduce.
- Aplicar un criterio explícito para decidir si un control vale su costo.
- Distinguir entre controles obligatorios, recomendados y opcionales según el contexto.

## La premisa: seguridad y operabilidad compiten.

Un sistema sin controles es fácil de operar y fácil de comprometer. Un sistema con todos los controles posibles es difícil de comprometer pero también difícil de operar, depurar y mantener. El punto óptimo no existe de forma universal: depende del modelo de amenaza, el tamaño del equipo, la madurez operativa y el valor del activo protegido.

La pregunta correcta no es «¿qué controles podemos añadir?» sino **«¿qué controles justifican su costo en este contexto?»**

Este cuaderno documenta las tensiones reales que aparecieron al construir el pipeline del curso.

## Inventario de *trade-offs* del curso.

Cada control del *pipeline* tiene un costo asociado. La tabla siguiente los hace explícitos.

| Control | Amenaza que mitiga | Costo operativo | Fragilidad introducida |
|---|---|---|---|
| `lifecycle.ignore_changes` en `image`, `env`, `volume_mounts` | Regresión de config por `terraform apply` | *Drift* silencioso: *Terraform* no detecta cambios en esos campos | Si *gcloud* y *Terraform* compiten por el mismo campo, el ganador depende de quién corrió último |
| `roles/artifactregistry.admin` para el *runner* de *Terraform* | *Terraform* necesita `getIamPolicy` para gestionar *bindings* IAM | Privilegios más amplios que los necesarios para el *deploy* real | La separación correcta (dos SAs) requiere más infraestructura |
| `--impersonate-service-account` en *smoke test* | *WIF* no emite *identity tokens* directamente | *Binding* adicional `serviceAccountTokenCreator`; más superficie *IAM* | Si el *binding* falta, el *smoke test* falla con error críptico |
| URL candidata con `---` construida manualmente | Permite *smoke test* en revisión sin tráfico | Frágil: si *Cloud Run* cambia el patrón de URL, el test falla silenciosamente | No existe *API* directa para obtener la URL del *tag* |
| `attribute_condition` con lista de dos *repos* | Evita que *repos* no autorizados obtengan tokens *WIF* | Cada *repo* nuevo requiere actualizar *Terraform* y volver a aplicar | Fácil de olvidar al añadir *repos* |
| `db.create_all()` en cada arranque | Garantiza esquema en prod sin migración explícita | Bloqueo breve en inicio; no detecta cambios de esquema destructivos | No es equivalente a un sistema de migraciones real |
| *Trivy* image scan con `exit-code: 1` | Bloquea *deploys* con CVEs conocidos HIGH/CRITICAL | Puede bloquear *deploys* legítimos por CVEs sin *fix* disponible | `ignore-unfixed: true` mitiga pero no elimina los falsos bloqueos |
| *Cosign* sign + *SBOM* attestation | Trazabilidad de cadena de suministro; detección de imágenes sustituidas | +3-5 min por *pipeline*; depende de *Sigstore*/*Rekor* (servicio externo) | Si *Rekor* no está disponible, el *pipeline* falla |
| *Blue/green* con `--no-traffic` y *smoke test* | Evita tráfico a revisiones rotas | El *deploy* tarda más; la revisión candidata queda activa aunque falle | Revisiones acumuladas en 0% que no se pueden borrar fácilmente |

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Control:
    nombre: str
    impacto_seguridad: int   # 1-5: cuánto reduce el riesgo real
    costo_operativo: int     # 1-5: complejidad, tiempo, dependencias externas
    fragilidad: int          # 1-5: probabilidad de romper el pipeline por sí mismo
    obligatorio_en_prod: bool

    @property
    def score(self) -> float:
        """Ratio beneficio/costo. Mayor es mejor."""
        return self.impacto_seguridad / ((self.costo_operativo + self.fragilidad) / 2)


controles = [
    Control("OIDC/WIF sin llaves estáticas",        5, 2, 1, True),
    Control("Trivy fs (dependencias)",               4, 1, 1, True),
    Control("Trivy config scan (Dockerfile)",        3, 1, 1, True),
    Control("Trivy image scan",                      4, 2, 2, True),
    Control("Blue/green + smoke test",               4, 3, 3, True),
    Control("Cosign sign + SBOM",                    3, 3, 2, False),
    Control("Cosign verify attestation",             2, 4, 3, False),
    Control("attribute_condition WIF (dos repos)",   3, 2, 2, True),
    Control("lifecycle.ignore_changes (3 campos)",   2, 2, 4, False),
    Control("impersonación en smoke test",           2, 3, 3, False),
]

controles.sort(key=lambda c: c.score, reverse=True)

print(f"{'Control':<40} {'Seg':>4} {'Costo':>6} {'Frag':>5} {'Score':>6} {'Obligatorio':>12}")
print("-" * 80)
for c in controles:
    oblig = "sí" if c.obligatorio_en_prod else "opcional"
    print(f"{c.nombre:<40} {c.impacto_seguridad:>4} {c.costo_operativo:>6} {c.fragilidad:>5} {c.score:>6.2f} {oblig:>12}")

## El problema del *runner* de *Terraform*: un caso de estudio.

Este curso encontró en producción una tensión que ilustra bien el problema:

1. El *pipeline* de *CI/CD* necesita `roles/artifactregistry.writer` para hacer `docker push`.
2. *Terraform* necesita `getIamPolicy` para gestionar los *bindings* *IAM* del mismo repositorio.
3. `getIamPolicy` no está en `writer` ni en `repoAdmin` (nivel *repo*) — solo en `roles/artifactregistry.admin` a nivel de proyecto.
4. La solución aplicada fue añadir `roles/artifactregistry.admin` al mismo SA que hace el *deploy*.

Esto viola el principio de mínimo privilegio: el SA de despliegue terminó con permisos de administración de toda la infraestructura de *Artifact Registry*.

**La solución correcta** es separar responsabilidades en dos SAs:

```
terraform-runner@...   ← solo corre terraform apply; permisos amplios de gestión
cicd-deployer@...      ← solo corre docker push y gcloud run deploy; permisos mínimos
```

Cada uno tiene su propio *WIF binding* restringido al *workflow* que lo necesita:

```hcl
# Solo el workflow de infra puede usar terraform-runner
member = "principalSet://.../attribute.job_workflow_ref/refs/heads/main:infra.yaml"

# Solo el workflow de deploy puede usar cicd-deployer
member = "principalSet://.../attribute.environment/prod"
```

**Por qué el curso no lo hace así:** añade un tercer SA, un tercer *WIF binding* y más variables en *GitHub* Environments. Para un proyecto de aprendizaje con un solo equipo, el costo operativo supera el beneficio.

**Cuándo sí separar:** en proyectos con múltiples equipos, cuando el SA de *deploy* es compartido entre *repos*, o cuando una auditoría exige demostrar mínimo privilegio real en el *pipeline* de producción.

## Marco de decisión: ¿vale este control su costo?

Antes de añadir un control al *pipeline*, responde estas cuatro preguntas:

### 1. ¿A qué amenaza concreta responde?

Un control sin amenaza nombrada es ruido. Si no puedes describir el ataque que previene, probablemente no lo necesitas.

### 2. ¿Cuál es la probabilidad y el impacto de esa amenaza en tu contexto?

| Impacto \\ Probabilidad | Baja | Media | Alta |
|---|---|---|---|
| **Alto** | Controles preventivos | Obligatorio | Obligatorio + redundancia |
| **Medio** | Opcional | Recomendado | Recomendado |
| **Bajo** | Ignorar | Documentar | Documentar |

### 3. ¿Qué rompe este control cuando falla?

Un control que bloquea el *pipeline* por razones no relacionadas con seguridad (dependencia externa caída, falso positivo, cambio de *API*) es un riesgo operativo. Debe tener:
- un procedimiento de *bypass* documentado para emergencias;
- alertas diferenciadas entre fallo de seguridad y fallo operativo.

### 4. ¿El equipo puede operar este control?

Un control que nadie entiende no es un control — es una trampa. Si el equipo no puede diagnosticar por qué falló, lo desactivará en la primera emergencia.

In [ ]:
def decision_control(
    amenaza_nombrada: bool,
    probabilidad: str,   # "baja" | "media" | "alta"
    impacto: str,        # "bajo" | "medio" | "alto"
    equipo_puede_operar: bool,
    tiene_bypass_documentado: bool,
) -> tuple[str, str]:
    if not amenaza_nombrada:
        return "descartar", "Sin amenaza nombrada, el control es ruido."

    matriz = {
        ("alto", "alta"): "obligatorio",
        ("alto", "media"): "obligatorio",
        ("alto", "baja"): "recomendado",
        ("medio", "alta"): "recomendado",
        ("medio", "media"): "recomendado",
        ("medio", "baja"): "opcional",
        ("bajo", "alta"): "documentar",
        ("bajo", "media"): "documentar",
        ("bajo", "baja"): "ignorar",
    }

    nivel = matriz.get((impacto, probabilidad), "revisar")

    notas = []
    if not equipo_puede_operar:
        notas.append("Bajar prioridad: equipo no puede operarlo.")
    if nivel in ("obligatorio", "recomendado") and not tiene_bypass_documentado:
        notas.append("Añadir bypass documentado para emergencias.")

    return nivel, " ".join(notas) if notas else "Sin observaciones adicionales."


casos = [
    ("Trivy image scan",          True, "media", "alto",  True,  True),
    ("Cosign verify-attestation", True, "baja",  "medio", False, False),
    ("Separar SA terraform/cicd", True, "baja",  "medio", True,  True),
    ("URL candidata con ---",     True, "alta",  "alto",  True,  False),
    ("Escaneo sin amenaza clara", False, "baja", "bajo",  True,  True),
]

print(f"{'Control':<30} {'Decisión':<14} Nota")
print("-" * 80)
for nombre, amenaza, prob, impacto, opera, bypass in casos:
    nivel, nota = decision_control(amenaza, prob, impacto, opera, bypass)
    print(f"{nombre:<30} {nivel:<14} {nota}")

## La curva de retornos decrecientes.

Los primeros controles eliminan los riesgos más grandes con el menor costo. A medida que se añaden controles, el riesgo residual disminuye más lentamente y el costo operativo crece más rápido.

```
Riesgo
residual
  │
  │●
  │  ●
  │    ●
  │      ●
  │        ●  ●  ●  ●
  └─────────────────────── Controles añadidos
```

El punto de inflexión varía por contexto:

- **Proyecto de aprendizaje**: los primeros 4-5 controles capturan el 80% del valor. Los siguientes son material pedagógico, no necesidad operativa.
- **Startup con un equipo pequeño**: hasta donde el equipo pueda operar con confianza. Un control mal entendido rompe más que protege.
- **Empresa con equipo de seguridad**: se pueden sostener más controles porque hay personas que los operan y monitorean activamente.
- **Infraestructura crítica regulada**: los controles son obligatorios independientemente del costo; el marco lo decide la regulación.

## Señales de que se endureció de más.

- El *pipeline* falla por razones no relacionadas con el código (dependencias externas, propagación de *IAM*, reglas de *Trivy* sin *fix*).
- El equipo desactiva controles en emergencias y no los reactiva.
- Nadie sabe por qué un paso específico existe o qué ataque previene.
- El tiempo de ciclo entre *commit* y *deploy* supera lo que el equipo tolera.
- Los falsos positivos son más frecuentes que los positivos reales.

## Señales de que se endureció de menos.

- Las credenciales de producción viven en archivos locales o variables de entorno permanentes.
- No existe trazabilidad entre un artefacto en producción y el *commit* que lo generó.
- Cualquier colaborador puede desplegar a producción sin revisión.
- No hay forma de detectar si una imagen fue modificada después del *build*.
- Un `terraform apply` accidental puede destruir infraestructura de producción sin confirmación.

## Controles del curso: clasificación final.

| Control | Clasificación | Justificación |
|---|---|---|
| *OIDC*/*WIF* sin llaves estáticas | **Obligatorio** | Las llaves estáticas son el vector más explotado en *pipelines* comprometidos |
| `attribute_condition` restringida al *repo* exacto | **Obligatorio** | Sin esto, cualquier *repo* de *GitHub* puede obtener tokens |
| *Trivy* `fs` en dependencias | **Obligatorio** | Primer filtro antes del *build*; costo casi nulo |
| *Trivy* config scan del *Dockerfile* | **Obligatorio** | Detecta configuraciones de imagen inseguras antes del *push* |
| *Trivy* image scan por *digest* | **Obligatorio** | Detecta CVEs en la imagen exacta que se va a desplegar |
| *Blue/green* con *smoke test* | **Obligatorio** | Sin esto, una revisión rota llega a producción sin detección |
| *Rollback* con nombre de revisión explícito | **Obligatorio** | `PREVIOUS` no existe en *Cloud Run*; sin esto el *rollback* falla silenciosamente |
| `lifecycle.ignore_changes` en *Terraform* | **Obligatorio** en este *stack* | Necesario por la coexistencia de *Terraform* y *gcloud* en el mismo recurso |
| *Cosign* sign + *SBOM* | **Recomendado** | Trazabilidad forense; costo aceptable; requiere *Sigstore* disponible |
| Separar SA *terraform*/*cicd* | **Recomendado** en prod real | Mínimo privilegio estricto; costo alto en este proyecto de aprendizaje |
| *Cosign* `verify-attestation` en *smoke test* | **Opcional** | Añade latencia; el valor marginal sobre la firma ya verificada es bajo |
| `--impersonate-service-account` en *smoke test* | **Necesario** dado el *stack* | No es una elección de *hardening* — es un requisito técnico de *WIF* |

## Cierre.

El endurecimiento no es un estado final — es una posición que se negocia continuamente entre riesgo, costo y capacidad operativa del equipo. Un *pipeline* con controles que nadie entiende es menos seguro que uno con menos controles bien comprendidos.

El criterio central del curso es:

> **Cada control debe poder justificarse con una amenaza nombrada, un costo explícito y un responsable que lo opere.**

Cuando un control no cumple los tres, es candidato a ser simplificado, documentado como deuda técnica, o eliminado.

## Ejercicio: auditoría de controles de tu *pipeline*.

Aplica el marco de decisión a **tres controles del pipeline del curso**. Para cada uno:

1. Nombra la amenaza concreta que mitiga.
2. Estima probabilidad e impacto en tu contexto real (no el del curso).
3. Decide si lo mantendrías como está, lo simplificarías o lo eliminarías.
4. Justifica en una línea.

Usa la celda siguiente para registrar tu análisis. Sustituye los valores de ejemplo por tu evaluación real.

In [ ]:
from dataclasses import dataclass


@dataclass
class AuditoriaControl:
    control: str
    amenaza: str
    probabilidad: str   # "baja" | "media" | "alta"
    impacto: str        # "bajo" | "medio" | "alto"
    decision: str       # "mantener" | "simplificar" | "eliminar"
    justificacion: str


# ── Completa con tu análisis real ──────────────────────────────────────────
mi_auditoria = [
    AuditoriaControl(
        control="Cosign verify-attestation",
        amenaza="Imagen sustituida post-firma en el registro",
        probabilidad="baja",
        impacto="alto",
        decision="simplificar",
        justificacion="La firma ya cubre el caso; verify-attestation añade 2 min por run sin "
                      "reducir riesgo adicional en un equipo de una persona.",
    ),
    AuditoriaControl(
        control="Trivy image scan con exit-code 1",
        amenaza="CVE conocido en imagen llega a producción",
        probabilidad="media",
        impacto="alto",
        decision="mantener",
        justificacion="Costo aceptable; ignore-unfixed evita bloqueos por CVEs sin parche.",
    ),
    AuditoriaControl(
        control="lifecycle.ignore_changes en Terraform",
        amenaza="terraform apply revierte imagen o secretos del pipeline",
        probabilidad="alta",
        impacto="alto",
        decision="mantener",
        justificacion="Sin este control, cualquier apply destruye la revisión activa.",
    ),
]
# ───────────────────────────────────────────────────────────────────────────

NIVEL = {
    ("alto", "alta"): "obligatorio",
    ("alto", "media"): "obligatorio",
    ("alto", "baja"): "recomendado",
    ("medio", "alta"): "recomendado",
    ("medio", "media"): "recomendado",
    ("medio", "baja"): "opcional",
    ("bajo", "alta"): "documentar",
    ("bajo", "media"): "documentar",
    ("bajo", "baja"): "ignorar",
}

print(f"{'Control':<35} {'Nivel sugerido':<16} {'Decisión':<12} Justificación")
print("─" * 100)
for a in mi_auditoria:
    nivel = NIVEL.get((a.impacto, a.probabilidad), "revisar")
    print(f"{a.control:<35} {nivel:<16} {a.decision:<12} {a.justificacion[:60]}")

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>